# Monte Carlo improvement research

Target: **over 40% wins with hidden-card information correctly restricted**. This is an objective, not an achieved result. Win rate is primary. Concurrent A* tests make timings unsuitable for controlled speed claims.

Work on `main`; edit only Monte Carlo implementation, its tests and its research artifacts. Do not overwrite A* or shared documentation changes. Development results stay out of the root README table.

## Starting evidence and actual algorithm

[Baseline](../../engine/logs/profiles/latest-players-1000-20261016T074956Z/summary.txt), commit `8fbfe34`: **156/1,000 (15.6%)**, 8.972 seconds/game, 214.29 moves/game, 1,000-move cap. These were independent random deals, not a matched control for this study.

The current player uses a flat root UCB allocator: 64 evaluations per eligible candidate, 128 DFS visits per sampled arrangement, rewards win=1/exhausted=0/cutoff=0.5. It samples every UNKNOWN from the planning identity pool without replacement and backtracks on one mutable candidate board. Final selection uses mean reward, then immediate win and existing board progress. Older random-playout descriptions are stale. Search currently uses legal-generation order and active-path cycle checks.

## Information and rules boundary

Every real decision starts from `planningCopy()`. Unseen stock and buried tableau identities must remain masked. Full-information DFS is permitted only on a newly sampled arrangement; real hidden identities must never reach the sampler. Known cards remain known across stock recycling. Determinization can still be optimistic about future information (strategy fusion); it is not a calibrated win probability.

[Local reproduction](../../../Solitaire-Repos/literature-reproduction/001-literature-reproduction/README.md) reports Lonelybot 48/100. Its stock knowledge, compressed actions and rules remain unaudited against ours. Treat its move ordering and transposition ideas as hypotheses, not evidence of a comparable target win rate. No external win rate is adopted as our own.

## Protocol

- First screen: 20 saved full deck orders, Java canonical suit/rank shuffle seeds 0–19, matching the A* screen. Save all decks before running. Compare their exact card orders to A*'s saved manifest.
- Each variant starts fresh on the same deck; a fixed sampling RNG seed makes reruns reproducible. Variant trajectories may consume different random draws; matched deals do not imply identical sampled worlds after divergence.
- Change one factor per arm: baseline; reveal-first ordering; 1,000 DFS visits; per-sample transpositions with incoming-move context. Defaults remain unchanged during screening.
- Validate every emitted command against real legal moves. Use the same 1,000-move cap and classify cap/quit as benchmark non-wins, not proofs of impossibility.
- Report paired wins gained/lost and per-deal results. A 20-deal screen cannot establish >40%. Freeze promising configurations before held-out evaluation; never tune on that held-out set. Confirm on larger independent batches before promoting a default.
- Use separate Monte Carlo artifact, snapshot, Gradle cache and build directories. Source hashes, exact commands, config and raw rows belong with each run.

## Hypotheses

H1: Reveal-first DFS ordering spends more of its visit budget on information/progress rather than tableau shuffles. Prior ten-deal diagnostics found more simulated solutions but did not establish a win-rate benefit over the current tie-break.

H2: Raising the DFS visit budget from 128 to 1,000, keeping 64 samples, resolves more cutoffs. This is intentionally more compute; earlier eight-sample/1,000-visit trials changed two budgets and do not answer this question.

H3: Avoid re-expanding the same sampled board with the same incoming move across sibling branches. Include incoming-move context because inverse-move pruning depends on it. Clear the cache between samples; preserve existing active-path cycle checks and undo boundaries.

## Run ledger

Artifacts: `experiments/runtime/artifacts/monte-carlo-research-20261017/`. Initial 20-deal screen complete; follow-up screening running. Held-out validation remains pending. Completed results and decisions are appended below; the code cells read live artifact status.


In [ ]:
from pathlib import Path
import csv, json
from collections import defaultdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine/build.gradle").exists())
ARTIFACTS = ROOT / "experiments/runtime/artifacts/monte-carlo-research-20261017"

def paired_report(path):
    groups = defaultdict(dict)
    with Path(path).open() as f:
        for row in csv.DictReader(f):
            groups[row["variant"]][int(row["seed"])] = row
    common = sorted(set.intersection(*(set(g) for g in groups.values())))
    base = groups["baseline"]
    for name, rows in groups.items():
        wins = sum(rows[s]["won"] == "true" for s in common)
        gained = sum(rows[s]["won"] == "true" and base[s]["won"] != "true" for s in common)
        lost = sum(rows[s]["won"] != "true" and base[s]["won"] == "true" for s in common)
        print(name, len(common), wins, "gained", gained, "lost", lost)
    return groups


## Implementation and validation — 17 October 2026

Added an explicit `RandomGenerator` constructor, opt-in reveal ordering and opt-in per-sample board/incoming-move transpositions. Default selection, rewards, sample count and visit limit are unchanged. No A* or shared engine files were edited. Snapshot starts at `8fbfe34` plus only the Monte Carlo research files; unrelated concurrent changes are excluded deliberately.

Isolated unit and Monte Carlo player suite: **244 passed, 1 existing skip, 0 failures**. New tests cover hidden-arrangement blindness of decisions and candidate rewards under all four ordering/cache combinations, context-sensitive transpositions, reset between samples, rollback and preservation of the move set under ordering.

All **20 saved deck orders match the A* screen exactly**. SHA-256 of the Monte Carlo manifest: `dcd31bd73ce6ba15ea9f382b582d83e7e65fcc4d2edf002b0e823cb3df42a750`. The file is saved before any games run. Fixed sampling seed: `4242` per fresh player.

### Commands

From `experiments/runtime/artifacts/monte-carlo-research-20261017/engine-snapshot`:

```sh
./gradlew --project-cache-dir ../gradle-cache -I ../isolated-build.gradle test --tests 'ai.games.unit.*' --tests ai.games.player.ai.MonteCarloPlayerTest --console=plain -Dorg.gradle.jvmargs='-Xms256m -Xmx2g'
```

The exact screen command, absolute output paths and process state are saved in `screen-20/metadata.json`. The runner uses `MonteCarloScreenTest`, `montecarlo.screen=true`, `montecarlo.games=20`, `montecarlo.seedStart=0`, `montecarlo.variants=baseline,order,visits,transpositions`, and `montecarlo.samplingSeed=4242`. `isolated-build.gradle` points Gradle output at this artifact tree, never `engine/build`. Each run retains full emitted command traces and game rows.

| Arm | Samples / eligible candidate | DFS visits | Ordering | Transpositions |
|---|---:|---:|---|---|
| baseline | 64 | 128 | generation | disabled |
| order | 64 | 128 | reveal first | disabled |
| visits | 64 | 1000 | generation | disabled |
| transpositions | 64 | 128 | generation | board + incoming move |

Each non-baseline arm changes one factor. The current screen is running. No partial result establishes a policy improvement.


In [ ]:
run = ARTIFACTS / "screen-20"
metadata = json.loads((run / "metadata.json").read_text())
print(metadata["state"])
print(" ".join(metadata["command"]))
if metadata["state"] == "completed":
    screen = paired_report(run / "games.csv")
else:
    print("Screen incomplete: do not interpret partial win rates as matched results.")


## Predeclared follow-up and held-out protocol

The artifact-local `pipeline.py` supervises the current screen. An arm must gain at least two wins over baseline on the 20 development deals to qualify. Among qualifying results, rank wins first, then fewer moves, then variant name. If none qualifies, stop and record that a new hypothesis is required; do not consume held-out deals.

For a qualifying parent, run a second development screen comparing that parent with each single remaining factor added. Keep the parent unless an addition gains at least two further wins. Freeze the selected configuration in `frozen-configuration.json` before inspecting any new deals. Compare frozen candidate versus baseline on **100 held-out deals, seeds 10000–10099**, with the same sampling seed and move cap. Do not tune on those outcomes. Record Wilson intervals and paired gains/losses; >40 wins on that small held-out batch would still require a larger untouched confirmation before claiming a settled >40% rate.

The supervisor appends completed results and findings to this notebook. It never edits player defaults, A* files, or either README. Exact process state is in `pipeline-status.json`; the initial command is `python3 experiments/runtime/artifacts/monte-carlo-research-20261017/pipeline.py`.


## H4: retain a solution once all cards are known

Development trace evidence, collected while the first screen was running: on deal 0, the 1,000-visit arm had **zero unknown identities at move 175** but was still at **14 foundation cards at move 350**. It eventually won at move 415. The baseline won at move 160. This suggests that discarding a known winning continuation can waste decisions even after uncertainty is gone. The trace audit is diagnostic and did not inspect held-out deals.

Added a separate opt-in `montecarlo.search.reuseKnownSolution` experiment. It records a winning DFS path only if the candidate has no unknown identities **before sampling**. It follows that path on subsequent real turns only after the usual observation validation and a legal-move check. A sampled world containing guesses cannot supply a retained plan. Default remains false.

A separate immutable snapshot/build/cache in `known-solution-study/` compares **baseline versus solution reuse**, with all other controls at their original values. It uses the identical saved 20 deck orders and sampling seed. This is one-factor development screening, independent of the original three-arm hypotheses; the original running snapshot is unchanged.

Validation: **250 passed, 1 existing skip, 0 failures**. Additional tests cover execution of a known winning path without another search, refusal to retain a path involving an unobserved card, and equal-observation blindness under all eight ordering/transposition/solution combinations. No A* files or shared build output were changed.

If solution reuse gains at least two wins on the 20 development deals, its already-fixed configuration will be compared against baseline on separate held-out seeds **11000–11099**. These are distinct from the original pipeline's held-out deals. Do not choose or retune a configuration using either held-out set. Larger untouched confirmation is still required for a settled >40% claim.


In [ ]:
study = ARTIFACTS / "known-solution-study"
print(json.loads((study / "pipeline-status.json").read_text()))
for stage in ["screen-20", "heldout-100"]:
    report = study / stage / "analysis.json"
    if report.exists():
        print(stage, json.loads(report.read_text()))
    else:
        print(stage, "not completed; no outcome claim")


## Known-solution screen: completed

| Policy | Wins / 20 | Paired wins gained | Paired wins lost | Average moves |
|---|---:|---:|---:|---:|
| Baseline | 5 | 0 | 0 | 195.00 |
| Reuse fully observed solution | 5 | 0 | 0 | 199.05 |

The same five deals were won. The 25% rate on these 20 selected development deals is not evidence of improvement over the separate 15.6% historical sample. This did **not** meet the +2-win development threshold, so the supervisor did not consume its held-out deals. Do not promote this flag on win-rate evidence. It may interact with a larger search budget, but that interaction is untested. The target is not achieved.

## H5: conservative foundation dominance inside DFS

Lonelybot's `state.rs` forces safe tableau foundation placements; its stock dominance uses a different, compressed access model. Copying a rule that forces every safe waste placement would be inappropriate: removing a waste card changes future draw-three access. Therefore this experiment forces **tableau-to-foundation** placements only and excludes withdrawals of conservatively safe foundation cards. Root candidate generation is unchanged. Waste placements remain ordinary choices.

The conservative card condition is: aces and twos; otherwise both opposite-colour foundations at least rank minus one, and the other same-colour foundation at least rank minus two. All checks use the current sampled board. This is a search-pruning hypothesis under our rules, not a claim that the external solver's rules match ours. Exhaustion remains relative to permitted pruning.

One-factor comparison: baseline versus `dominance`, with 64 evaluations per eligible candidate, 128 DFS visits, generation ordering, transpositions off and retained-solution mode off. Same 20 saved deals and sampling seed 4242. Separate immutable snapshot/build/cache/output: `foundation-dominance-study/`.

Predeclared qualification remains +2 development wins. If qualified, freeze the flag and evaluate against baseline on 100 untouched deals, seeds **12000–12099**, separate from both earlier held-out ranges. No tuning on that set. This study is pending validation/start at this entry; its metadata and status cells below provide the current state.


In [ ]:
study = ARTIFACTS / "foundation-dominance-study"
status = study / "pipeline-status.json"
print(json.loads(status.read_text()) if status.exists() else "Not started")
for stage in ["screen-20", "heldout-100"]:
    report = study / stage / "analysis.json"
    if report.exists(): print(stage, json.loads(report.read_text()))


### H5 validation and launch

**261 passed, 1 existing skip, 0 failures** in the isolated unit and Monte Carlo player suite. Hidden-arrangement blindness is tested across all 16 combinations of the opt-in controls. Tests also distinguish safe tableau placements from waste placements, check the conservative rank/colour thresholds, and check removal of safe higher-rank foundation withdrawals.

The first withdrawal test used an ace; it correctly failed its setup assertion because the existing move generator already omits ace/two withdrawals. The fixture was corrected to a safe three with all required lower foundations. The failure and corrected run are both retained; no production rule was changed to satisfy the test.

The dominance screen and conditional held-out supervisor are now running independently. Defaults remain unchanged. No result has yet established the >40% target.


## Development protocol amendment before any held-out runs

The original 20-deal screen is still running. Its code and arms remain unchanged. The completed baseline/solution screen showed no standalone win gain, but the larger-budget trace demonstrated the interaction worth testing: many solved evaluations can coexist with hundreds of real-game detours after all cards are known.

The conditional follow-up screen therefore adds **solution reuse** as one more possible single-factor addition to the winning original arm, alongside ordering, visits and transpositions as applicable. It still permits only one follow-up round, requires +2 additional wins to change the parent, and freezes the resulting configuration before any held-out results. This amendment uses development evidence only; no held-out outcomes have been inspected.

Follow-up and held-out stages use the separate, tested `known-solution-study/engine-snapshot`, with safe-foundation dominance disabled. The parent is rerun and must reproduce both wins and move counts exactly against the original screen before child comparisons are accepted. Source hashes and exact per-stage snapshot paths are saved. The original supervisor was restarted while waiting for the unchanged first screen; the old script and status are preserved as `pipeline-v1.py` and `pipeline-status-v1.json`. No game run was interrupted.


## Conservative dominance screen: completed

| Policy | Wins / 20 | Paired wins gained | Paired wins lost | Average moves |
|---|---:|---:|---:|---:|
| Baseline | 5 | 0 | 0 | 195.00 |
| Conservative DFS foundation dominance | 5 | 0 | 0 | 192.35 |

Again the same five deals were won. This failed the +2-win qualification threshold, so no held-out deals were consumed and the default remains off. The tiny move-count difference is not a win-rate improvement. The separate ordering/budget/transposition screen continues.

The fully-known detour observation is reproducible from `audit-prefixes/visits-0.commands.txt` (the first 350 emitted moves), the saved deal manifest, `TraceAudit.java` and `trace-audit.txt`. It is a development diagnosis, not a held-out finding.



## screen-20 — completed 2026-10-16T21:43:18.882425+00:00

| Variant | Wins | Games | Gained vs baseline | Lost vs baseline | Avg moves | Avg seconds* |
|---|---:|---:|---:|---:|---:|---:|
| baseline | 5 | 20 | 0 | 0 | 195.00 | 8.874 |
| order | 9 | 20 | 5 | 1 | 240.75 | 12.566 |
| visits | 8 | 20 | 4 | 1 | 232.80 | 67.732 |
| transpositions | 4 | 20 | 0 | 1 | 193.55 | 8.756 |

*Concurrent work makes timings descriptive only. Raw games, command traces, exact command and metadata: `experiments/runtime/artifacts/monte-carlo-research-20261017/screen-20/`.



### Follow-up selection
Parent: `order`. Each follow-up arm adds exactly one remaining factor to this parent on the same 20 development deals. This is continued screening, not validation.


## Initial screening finding

Reveal-first ordering won **9/20 (45%)**, versus baseline **5/20 (25%)**. It retained four of the five baseline wins, gained five new wins, and lost one. The 95% Wilson interval for its win rate is approximately **25.8%–65.8%**. This is a selected development result, not evidence that the population win rate exceeds 40%.

The one-factor larger visit budget won 8/20; transpositions won 4/20. Standalone known-solution reuse and conservative foundation dominance each matched baseline at 5/20. No default changed.

The active follow-up compares `order`, `order-solution`, `order-transpositions`, and `order-visits` on the same saved 20 deals. Parent wins and move counts must reproduce before accepting these comparisons. The predeclared supervisor then freezes a configuration and runs baseline/candidate on 100 untouched deals. It records results here automatically and stops on run failures or reproduction mismatches.

Current source matches the tested foundation-study snapshot: **261 passed, 1 existing skip, 0 failures**. A* files and the canonical root README table were not edited by this work.



## followup-20 — completed 2026-10-16T22:13:34.690051+00:00

| Variant | Wins | Games | Gained vs order | Lost vs order | Avg moves | Avg seconds* |
|---|---:|---:|---:|---:|---:|---:|
| order | 9 | 20 | 0 | 0 | 240.75 | 12.088 |
| order-solution | 9 | 20 | 0 | 0 | 238.85 | 11.586 |
| order-transpositions | 7 | 20 | 0 | 2 | 250.25 | 11.768 |
| order-visits | 6 | 20 | 0 | 3 | 248.40 | 54.788 |

*Concurrent work makes timings descriptive only. Raw games, command traces, exact command and metadata: `experiments/runtime/artifacts/monte-carlo-research-20261017/followup-20/`.



### Configuration frozen before held-out evaluation
Candidate: `order`; 100 fresh deals (seeds 10000–10099), paired with baseline. Sampling seed remains 4242; move cap remains 1,000. No held-out result will be used to change this configuration.



## heldout-100 — completed 2026-10-16T22:47:05.652594+00:00

| Variant | Wins | Games | Gained vs baseline | Lost vs baseline | Avg moves | Avg seconds* |
|---|---:|---:|---:|---:|---:|---:|
| baseline | 15 | 100 | 0 | 0 | 216.28 | 9.177 |
| order | 34 | 100 | 20 | 1 | 224.78 | 10.844 |

*Concurrent work makes timings descriptive only. Raw games, command traces, exact command and metadata: `experiments/runtime/artifacts/monte-carlo-research-20261017/heldout-100/`.



### Held-out conclusion
The frozen candidate did not exceed 40% on the held-out sample. The target remains unachieved. Do not tune this candidate on these held-out results. No experimental default was automatically promoted.


## Round 2: cutoff scoring and foundation dominance with ordering

The frozen ordering policy won **34/100**, versus baseline **15/100**, on the completed held-out set (seeds 10000–10099). These deals are now excluded from tuning and future validation. The >40% target remains unmet. Ordering becomes the reference policy for this round; no production defaults change.

**H6:** the current unresolved-search reward of 0.5 may favour positions whose search merely hits the visit budget. Test cutoff rewards **0.0** and **0.25**, independently, with ordering enabled. Wins remain 1, exhausted permitted searches remain 0, and rejection remains -1. A cutoff score is an experimental ranking choice, not a claim that the position is lost or has a calibrated win probability. Keep 128 visits and 64 evaluations per eligible root candidate.

**H7:** conservative tableau foundation dominance may help ordered DFS even though it did not improve baseline on its own. Compare `dominance-order` against `order` with the default 0.5 cutoff reward. Waste placements are still never forced. No Lonelybot stock/rules equivalence is assumed.

Predeclared screen: same saved 20 development deals, seeds 0–19; arms `order`, `order-cutoff0`, `order-cutoff25`, `dominance-order`. Each changes one factor from the reference. Require at least **two extra wins** to qualify, then rank by wins, fewer total moves, variant name. First check that reference wins and move counts reproduce exactly from the earlier screen. Freeze at most one qualifying candidate before paired validation on **100 untouched deals, seeds 20000–20099**, against ordering. Do not tune on held-out results. If none qualifies, stop and record the negative result without consuming held-out deals.

Artifacts and immutable snapshot: `experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/`. Saved decks, source hashes, `protocol.json`, exact Gradle commands, logs and per-game traces stay there. A* and shared build files are untouched.


### Round 2 validation and launch

Isolated unit tests and Monte Carlo functional tests: **270 passed, 1 existing skip, 0 failures**. Added checks for cutoff scores, unchanged winning rewards, board/history restoration, invalid configurations and hidden-card blindness under the new scores. Source hashes and `unit-tests.log` are saved with the study.

Commands from the repository root:

```sh
python3 experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/launch.py screen-20 order,order-cutoff0,order-cutoff25,dominance-order 0 20
python3 experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/pipeline.py
```

The second command is already running as a detached supervisor; do not start a duplicate. Exact live state and process identifiers are in `pipeline-status.json` and each run's `metadata.json`. Completed tables and findings are appended here automatically. All experimental defaults remain unchanged.


In [ ]:
study = ARTIFACTS / "cutoff-reward-study"
print(json.loads((study / "pipeline-status.json").read_text()))
for stage in ["screen-20", "heldout-100"]:
    result = study / stage / "analysis.json"
    if result.exists():
        print(stage, json.loads(result.read_text()))



## screen-20 — completed 2026-10-17T00:07:57.474057+00:00

| Variant | Wins | Games | Gained vs order | Lost vs order | Avg moves | Avg seconds* |
|---|---:|---:|---:|---:|---:|---:|
| order | 9 | 20 | 0 | 0 | 240.75 | 12.465 |
| order-cutoff0 | 6 | 20 | 0 | 3 | 232.60 | 10.532 |
| order-cutoff25 | 8 | 20 | 0 | 1 | 182.55 | 9.030 |
| dominance-order | 10 | 20 | 1 | 0 | 245.05 | 12.310 |

*Concurrent work makes timings descriptive only. Raw games, command traces, exact command and metadata: `experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/screen-20/`.



### Round 2 screening conclusion
No tested change cleared the predeclared +2-win gain over ordering. No held-out deals were consumed. Defaults remain unchanged; the >40% objective still needs another hypothesis.


## Independent validation of the two strongest configurations

At the user's request, independently evaluate **ordering alone** versus **ordering plus conservative foundation pruning**, despite the latter missing the automated +2-win development threshold (10/20 versus 9/20). The earlier screening result and stopping decision remain recorded. The comparison was selected before launch and no validation outcomes were inspected to choose it.

Fixed configuration: 128 DFS visits, 64 samples per eligible root candidate, cutoff reward 0.5, reveal-first ordering enabled, transpositions and solution retention disabled. Only foundation dominance differs. Hidden information remains restricted through planning copies and sampled identities. Use the previously unconsumed **100 saved deals, seeds 20000–20099**, and sampling seed 4242; 1,000 real moves per game. The harness saves all deck orders before playing and rotates variant order by deal. Run the full batch without tuning or outcome-based early stopping.

Record wins, paired wins gained/lost, 95% Wilson intervals and descriptive timing. The previous 100 validation deals (10000–10099) remain excluded. No settings will be automatically promoted. A* code, shared build output and the root README remain outside this work.

The tested snapshot's source/build inputs match the recorded hashes. A verification attempt initially included the mutable `logs/game.log`, so the protocol record and reporting supervisor were completed after launch, before inspecting any outcomes; runtime logs were excluded from the corrected verification. The game run's configuration was already fixed by its exact launch command. Protocol: `experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/independent-validation-protocol.json`.

Commands (already launched; do not duplicate):

```sh
python3 experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/launch.py heldout-100 order,dominance-order 20000 100
python3 experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/validation.py
```

The detached validator records `validation-status.json` separately from the completed screening pipeline and appends the finished result here. Raw per-deal rows, traces and exact Gradle invocation are in the study's `heldout-100/` directory.


In [ ]:
study = ARTIFACTS / "cutoff-reward-study"
print(json.loads((study / "validation-status.json").read_text()))
result = study / "heldout-100" / "analysis.json"
if result.exists():
    print(json.loads(result.read_text()))



## heldout-100 — completed 2026-10-17T01:21:35.478511+00:00

| Variant | Wins | Games | Gained vs order | Lost vs order | Avg moves | Avg seconds* |
|---|---:|---:|---:|---:|---:|---:|
| order | 25 | 100 | 0 | 0 | 198.25 | 8.905 |
| dominance-order | 30 | 100 | 15 | 10 | 198.73 | 9.881 |

*Concurrent work makes timings descriptive only. Raw games, command traces, exact command and metadata: `experiments/runtime/artifacts/monte-carlo-research-20261017/cutoff-reward-study/heldout-100/`.



### User-requested independent validation: conclusion
The foundation-pruning candidate did not exceed 40 wins on this fresh 100-deal sample. The >40% target remains unachieved. Paired net win gain versus ordering: 5/100. Neither defaults nor the canonical README were changed. These deals are now excluded from future tuning and validation.
